# Agentic Ai

## Create an Agent to plan, write and edit the content for a blogpost

### Dependencies

We will use

1. CrewAi
2. Ollama
3. LLama 3, although by default CrewAi uses OpenAi GPT3.5 or GPT4

Ollama is an open-source app that lets you run, create, and share large language models locally with a command-line interface

Download Ollama and install it on Windows.
Go to Ollama and download .exe file: https://ollama.com

You have the option to use the default model save path, typically located at:
C:\Users\your_user\.ollama

Then download the llama3 model from the command prompt:

In [15]:
#!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

### Set up the LLM as Llama3

Create a ModelFile in your project directory:

Run the following command in the command prompt

In [ ]:
from crewai import Agent, Task, Crew, LLM
import os

# Required by some OpenAI-compatible clients; any non-empty value works for Ollama
os.environ["GEMINI_API_KEY"] = "aaaa" # Poner aqui su API key

llm = LLM(
    model="gemini/gemini-2.5-pro",
    temperature=0.7
)

In [ ]:
# from crewai import Agent, Task, Crew
# from langchain_openai import ChatOpenAI
# import os
# os.environ["OPENAI_API_KEY"] = "NA"
# 
# llm = ChatOpenAI(
#     model = "gemini/gemini-2.5-flash",
#     base_url = "http://localhost:11434/v1")

#### Agent Attributes

- Role: Defines the agent’s function within the crew. It determines the kind of tasks the agent is best suited for.

- Goal: The individual objective that the agent aims to achieve. It guides the agent’s decision-making process.

- Backstory: Provides context to the agent’s role and goal, enriching the interaction and collaboration dynamics.

- LLM :(optional)Represents the language model that will run the agent. It dynamically fetches the model name from the OPENAI_MODEL_NAME environment variable, defaulting to "gpt-4" if not specified.

- Tools :(optional)Set of capabilities or functions that the agent can use to perform tasks. Expected to be instances of custom classes compatible with the agent's execution environment. Tools are initialized with a default value of an empty list.

- Function Calling LLM :(optional)Specifies the language model that will handle the tool calling for this agent, overriding the crew function calling LLM if passed. Default is None.

- Max Iter :(optional)The maximum number of iterations the agent can perform before being forced to give its best answer. Default is 25.

- Max RPM :(optional)The maximum number of requests per minute the agent can perform to avoid rate limits. It's optional and can be left unspecified, with a default value of None.

- max_execution_time :(optional)Maximum execution time for an agent to execute a task It's optional and can be left unspecified, with a default value of None, menaning no max execution time

- Verbose: (optional)Setting this to True configures the internal logger to provide detailed execution logs, aiding in debugging and monitoring. Default is False.

- Allow Delegation: (optional)Agents can delegate tasks or questions to one another, ensuring that each task is handled by the most suitable agent. Default is True.

- Step Callback: (optional)A function that is called after each step of the agent. This can be used to log the agent's actions or to perform other operations. It will overwrite the crew step_callback.

- Cache: (optional)Indicates if the agent should use a cache for tool usage. Default is True

#### Content Planner Agent

In [18]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic} in 'https://medium.com/'."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "You have to prepare a detailed "
              "outline and the relevant topics and sub-topics that has to be a part of the"
              "blogpost."
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    llm=llm,
    allow_delegation=False,
 verbose=True
)

#### Content Writer Agent

In [19]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic} in 'https://medium.com/'. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    llm=llm,
    verbose=True
)

#### Content Editor Agent

In [20]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization 'https://medium.com/'. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    llm=llm,
    allow_delegation=False,
    verbose=True
)

#### Task
Collaborative, require multiple agents to work together. Managed through the task properties and orchestrated by the Crew’s process, enhancing teamwork and efficiency.

Create planner task

In [21]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

Create Writer Task

In [22]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
  "3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

Create Editor Task

In [23]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

#### Creating the Crew of agents

- Pass the tasks to be performed by those agents.

Note: For this simple example, the tasks will be performed sequentially (i.e they are dependent on each other), so the order of the task in the list matters.

verbose=2 allows you to see all the logs of the execution.

In [ ]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True
)

#### Run the crew

In [ ]:
inputs = {"topic":"Comparative study of LangGraph, Autogen and Crewai for building multi-agent system."}
result = crew.kickoff(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d8349bd4-fb2a-431a-8e9c-855ee0e170fe                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Comparative study of LangGraph,     │
│  Autogen and Crewai for building multi-agent system..                                                           │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  ID: 4c51f5f5-cc17-4744-b8d8-094d4cd50bb4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Comparative study of LangGraph,     │
│  Autogen and Crewai for building multi-agent system..                                                           │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here is the complete content plan.                                                                             │
│                                                                                                                 │
│  ***                                                                                                            │
│                                                                                                                 │
│  ### **Content Plan: Comparative Study of LangGraph, Autogen, and CrewAI**                                      │
│                                                                                                                 │
│  **Blog Post Title:** LangGraph vs. Autogen vs. CrewAI: Choosing the Right Framework for Your Multi-Agent       │
│  System in 2024                                                                                                 │
│                                                                                                                 │
│  **Target Platform:** Medium.com (targeting publications like "Towards Data Science," "Better Programming," or  │
│  "The Startup")                                                                                                 │
│                                                                                                                 │
│  **Author Persona:** An experienced AI Engineer/Developer Advocate sharing practical insights and guidance.     │
│                                                                                                                 │
│  **Goal of the Article:** To demystify the multi-agent framework landscape by providing a clear, comparative    │
│  analysis of LangGraph, Autogen, and CrewAI. The reader should finish the article with a strong understanding   │
│  of which tool is best suited for their specific project needs.                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### **1. Latest Trends, Key Players & Noteworthy News**                                                        │
│                                                                                                                 │
│  *   **Overarching Trend:** The industry is rapidly moving beyond single-prompt LLM applications towards        │
│  sophisticated, multi-step "agentic workflows." These systems can plan, execute tasks, use tools, and           │
│  collaborate to solve complex problems autonomously.                                                            │
│  *   **Key Challenge Addressed:** A primary bottleneck in early agent frameworks was managing state and         │
│  creating non-linear, cyclical workflows (e.g., reflection, self-correction). The frameworks in this            │
│  comparison are direct responses to this challenge.                                                             │
│  *   **Key Players & Their Angle:**                                                                             │
│      *   **LangChain (LangGraph):** A major player in t

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Comparative study of LangGraph,     │
│  Autogen and Crewai for building multi-agent system..                                                           │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # LangGraph vs. Autogen vs. CrewAI: Choosing the Right Framework for Your Multi-Agent System in 2024           │
│                                                                                                                 │
│  ### A deep-dive into the paradigms, code, and use cases of today's leading AI agent orchestration frameworks.  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  We've moved past simple chatbots. The new frontier is autonomous AI agents that can collaborate, reason, and   │
│  execute complex tasks. We are no longer just prompting a large language model; we are building and managing    │
│  digital workforces. This paradigm shift has led to an explosion of powerful tools for **AI agent               │
│  orchestration**, but with it comes a paradox of choice. A flood of frameworks—LangGraph, Autogen, and          │
│  CrewAI—has emerged, each with a different philosophy, making it difficult for developers to know where to      │
│  start.                                                                                                         │
│                                                                                                                 │
│  This article cuts through the noise. We'll provide a direct, head-to-head comparison, breaking down each       │
│  framework's core concepts, strengths, and weaknesses. By the end, you'll have a clear mental model to help     │
│  you choose the right tool for your job, whether you're building a simple proof-of-concept or a                 │
│  production-grade agentic system. To set the stage, let's briefly introduce the contenders for building         │
│  **autonomous AI agents**:                                                                                      │
│                                                                                                                 │
│  *   **LangGraph:** The State Machine for fine-grained control.                                                 │
│  *   **Autogen:** The Conversation-Driven Collaborator for dynamic interactions.                                │
│  *   **CrewAI:** The Role-Playing Team Orchestrator for rapid development.                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### A Quick Primer: Why Multi-Agent Systems?                                                                   │
│                                                                                                                 │
│  Before we dive into the frameworks, let's clarify what we mean by a multi-agent system in the context of       │
│  LLMs. In short, it is a system where multiple specialized AI agents work together to achieve a common goal     │
│  that would be too complex or inefficient for a single 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d8349bd4-fb2a-431a-8e9c-855ee0e170fe                                                                       │
│  Final Output: # LangGraph vs. Autogen vs. CrewAI: Choosing the Right Framework for Your Multi-Agent System in  │
│  2024                                                                                                           │
│                                                                                                                 │
│  ### A deep-dive into the paradigms, code, and use cases of today's leading AI agent orchestration frameworks.  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  We've moved past simple chatbots. The new frontier is autonomous AI agents that can collaborate, reason, and   │
│  execute complex tasks. We are no longer just prompting a large language model; we are building and managing    │
│  digital workforces. This paradigm shift has led to an explosion of powerful tools for **AI agent               │
│  orchestration**, but with it comes a paradox of choice. A flood of frameworks—LangGraph, Autogen, and          │
│  CrewAI—has emerged, each with a different philosophy, making it difficult for developers to know where to      │
│  start.                                                                                                         │
│                                                                                                                 │
│  This article cuts through the noise. We'll provide a direct, head-to-head comparison, breaking down each       │
│  framework's core concepts, strengths, and weaknesses. By the end, you'll have a clear mental model to help     │
│  you choose the right tool for your job, whether you're building a simple proof-of-concept or a                 │
│  production-grade agentic system. To set the stage, let's briefly introduce the contenders for building         │
│  **autonomous AI agents**:                                                                                      │
│                                                                                                                 │
│  *   **LangGraph:** The State Machine for fine-grained control.                                                 │
│  *   **Autogen:** The Conversation-Driven Collaborator for dynamic interactions.                                │
│  *   **CrewAI:** The Role-Playing Team Orchestrator for rapid development.                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### A Quick Primer: Why Multi-Agent Systems?                                                                   │
│                                                                                                                 │
│  Before we dive into the frameworks, let's clarify what we mean by a multi-agent system in the context of       │
│  LLMs. In short, it is a system where multiple special

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



┌───────────────────────────── Execution Traces ──────────────────────────────┐
│                                                                             │
│  🔍 Detailed execution traces are available!                                │
│                                                                             │
│  View insights including:                                                   │
│    • Agent decision-making process                                          │
│    • Task execution flow and timing                                         │
│    • Tool usage details                                                     │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
Would you like to view your execution traces? [y/N] (20s timeout): 

#### Response

Let's see the response in the log of execution

#### Display the result

In [30]:
from IPython.display import Markdown,display
display(Markdown(result.raw))

# LangGraph vs. Autogen vs. CrewAI: Choosing the Right Framework for Your Multi-Agent System in 2024

### A deep-dive into the paradigms, code, and use cases of today's leading AI agent orchestration frameworks.

---

We've moved past simple chatbots. The new frontier is autonomous AI agents that can collaborate, reason, and execute complex tasks. We are no longer just prompting a large language model; we are building and managing digital workforces. This paradigm shift has led to an explosion of powerful tools for **AI agent orchestration**, but with it comes a paradox of choice. A flood of frameworks—LangGraph, Autogen, and CrewAI—has emerged, each with a different philosophy, making it difficult for developers to know where to start.

This article cuts through the noise. We'll provide a direct, head-to-head comparison, breaking down each framework's core concepts, strengths, and weaknesses. By the end, you'll have a clear mental model to help you choose the right tool for your job, whether you're building a simple proof-of-concept or a production-grade agentic system. To set the stage, let's briefly introduce the contenders for building **autonomous AI agents**:

*   **LangGraph:** The State Machine for fine-grained control.
*   **Autogen:** The Conversation-Driven Collaborator for dynamic interactions.
*   **CrewAI:** The Role-Playing Team Orchestrator for rapid development.

---

### A Quick Primer: Why Multi-Agent Systems?

Before we dive into the frameworks, let's clarify what we mean by a multi-agent system in the context of LLMs. In short, it is a system where multiple specialized AI agents work together to achieve a common goal that would be too complex or inefficient for a single agent to handle. This approach is a cornerstone of creating sophisticated **agentic workflows in Python**.

Think of it like building a house. You don't hire one person who "kind of" knows everything. You hire a specialized crew: an architect to design the plans, a plumber to handle the pipes, and an electrician to wire the building. AI agents work the same way. By breaking down a complex problem, you can assign specific roles and tools to different agents. This decomposition leads to several key advantages, including more robust task handling, the ability to assign specialized skills (like code execution or web browsing), and emergent problem-solving capabilities that arise from agent collaboration.

---

### The Contenders: A High-Level Overview

Each of these three frameworks offers a unique perspective on how to orchestrate collaboration between AI agents. Understanding their core philosophy is the first step in deciding which one fits your needs.

#### LangGraph: The Control Freak's Dream

Built by the team behind the ubiquitous LangChain library, LangGraph is their answer to building more robust and stateful agents. Its core idea is to model agentic workflows as a state machine or a graph. You explicitly define nodes, which represent a function or a tool (the "workers"), and edges, which represent the logic that directs the flow from one node to the next based on the current state.

The key feature of LangGraph is its **explicit state management**. A central state object is passed between nodes, and each node can modify it. This gives you unparalleled, fine-grained control over the entire process and makes debugging complex interactions significantly easier because you can trace the state at every step. Think of it as a detailed flowchart for your AI system, making it perfect for complex, cyclical processes that require reflection, modification, and precise **control flow**, such as "Plan -> Execute -> Get Feedback -> Revise Plan."

#### Autogen: The Power of Conversation

Hailing from Microsoft Research, Autogen proposes that agent collaboration is best modeled as a conversation. In this paradigm, agents are "conversable" entities that interact with each other in a group chat-like environment to solve problems. The developer's job is to define the agents, their capabilities, and the rules of engagement for their conversation.

Autogen’s primary strength lies in its **flexible conversation patterns**. You can design a wide variety of interaction models, from a simple two-agent chat to a complex, hierarchical discussion with automated group chats. It also has first-class support for **human-in-the-loop** integration, allowing a person to seamlessly join the conversation, provide feedback, and guide the agents. It’s like a highly configurable Slack workspace for AI agents, making it an excellent choice for academic research, complex simulations, and scenarios requiring dynamic agent interactions.

#### CrewAI: The Fast and Focused Team

CrewAI has rapidly gained popularity due to its developer-first approach and intuitive design. It abstracts away much of the underlying complexity of agent interaction by focusing on a simple, powerful metaphor: a crew of role-playing agents working together on a set of tasks. It's built on top of LangChain but provides a higher-level API that simplifies the process of building multi-agent systems.

The standout feature of CrewAI is its **high-level abstraction and simplicity**. You define Agents with specific `roles`, `goals`, and `backstories`, assign them sequential or parallel `Tasks`, and then launch the `Crew` to execute the workflow. The framework handles the intricate details of agent communication and task handoff for you. It's akin to being a manager who defines job roles and project goals, then trusts the team to figure out the execution details. This makes CrewAI ideal for rapidly building task-oriented applications, prototyping ideas, and for developers who value a declarative, easy-to-understand API over granular control.

---

### The Deep Dive: A Head-to-Head Comparison

To truly understand the differences, let's compare these frameworks across several key dimensions. This comparison should help you map your project requirements to the right tool.

| Feature               | LangGraph                                        | Autogen                                             | CrewAI                                                  |
| --------------------- | ------------------------------------------------ | --------------------------------------------------- | ------------------------------------------------------- |
| **Core Paradigm**     | State Machine / Graph                            | Conversational Agents                               | Role-Playing Crew                                       |
| **Learning Curve**    | Moderate to High                                 | High                                                | Low                                                     |
| **Flexibility/Control** | Very High                                        | Very High                                           | Moderate                                                |
| **State Management**  | Explicit & Centralized                           | Implicit (managed via conversation history)         | Implicit (managed by the framework)                     |
| **Human-in-the-Loop** | Highly Customizable (add a node for human input) | Natively Integrated (via `UserProxyAgent`)          | Simple (via task definition) but less flexible          |
| **Code Style**        | Imperative / Procedural (Define nodes & edges)   | Event-Driven / Conversational (Define agents & chat) | Declarative (Define roles, tasks, and crew)             |
| **Debugging**         | Easier (Traceable state & graph execution)       | Can be complex (Tracing long conversations)         | Abstracted away (harder to debug underlying interactions) |

#### Elaboration on Key Points

The core trade-off between these frameworks is **Flexibility vs. Simplicity**. CrewAI gives you incredible development speed by providing a high-level API, but this comes at the cost of granular control. If your application fits its role-playing model, you can build a functional system in minutes. LangGraph sits at the opposite end of the spectrum; it demands more boilerplate code to define the state, nodes, and edges, but in return, it grants you absolute control over every step of the process. Autogen offers immense conversational flexibility but requires you to adopt its specific, conversation-centric way of thinking about agent interaction.

**State management** is another critical differentiator. LangGraph's explicit, centralized state is a game-changer for building production-ready systems. Knowing the exact state of your workflow at any given moment is invaluable for **debugging agents** and ensuring reliability in long-running, complex tasks. In contrast, Autogen and CrewAI manage state implicitly through conversation history or internal mechanisms. While this simplifies development, it can make it harder to trace and debug when things go wrong in a complex interaction.

Finally, the **learning curve** is a practical consideration. A developer new to agentic systems can get a CrewAI application running in under an hour, thanks to its clear and declarative API. Autogen and LangGraph have steeper curves because they expose more low-level concepts. Autogen requires understanding its agent hierarchy and conversation patterns, while LangGraph requires a solid grasp of graph theory and state machines.

---

### Let's See the Code: A Practical Example

Theory and tables are useful, but nothing makes the differences clearer than seeing code. Let's tackle a practical task: "Create a market research report on the future of AI-powered code generation tools." This requires at least two agents: a `Researcher` to find relevant information and a `Writer` to synthesize it into a coherent report. The following snippets are simplified for clarity, but they perfectly illustrate the distinct developer experience and architectural philosophy of each framework.

#### CrewAI Snippet

Notice the high-level, declarative nature. You define *who* the agents are and *what* their tasks are, and the `Crew` handles the orchestration.

```python
# Emphasize the simplicity and readability
from crewai import Agent, Task, Crew

researcher = Agent(
  role='Market Researcher',
  goal='Find the latest trends in AI-powered code generation tools',
  backstory='An expert market researcher skilled in finding and analyzing data.',
  verbose=True
)
writer = Agent(
  role='Tech Report Writer',
  goal='Write a compelling 500-word report on AI code generation tools',
  backstory='A professional tech writer known for clear and concise analysis.',
  verbose=True
)

research_task = Task(description='Find and summarize 3 recent articles...', agent=researcher)
write_task = Task(description='Write a 500-word report based on the research findings.', agent=writer)

report_crew = Crew(
    agents=[researcher, writer], 
    tasks=[research_task, write_task],
    verbose=2
)
result = report_crew.kickoff()
```

#### Autogen Snippet

Here, the focus is on setting up the agents and initiating a conversation. The `UserProxyAgent` acts as the orchestrator, kicking off the task with a message.

```python
# Emphasize the conversation setup
import autogen

config_list = autogen.config_list_from_json("OAI_CONFIG_LIST")
llm_config = {"config_list": config_list}

researcher = autogen.AssistantAgent(
    name="Researcher",
    system_message="You are a market researcher. Find information and provide it.",
    llm_config=llm_config
)
writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a tech writer. Take research and write a report.",
    llm_config=llm_config
)
user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    code_execution_config=False,
    human_input_mode="NEVER"
)

user_proxy.initiate_chat(
    researcher,
    message="Create a market research report on AI code generation tools, then pass it to the writer."
)
```

#### LangGraph Snippet

This example highlights the explicit structure. We define a state dictionary and functions (nodes) that operate on that state. The graph structure dictates the flow from research to writing.

```python
# Emphasize the graph structure and state
from typing import TypedDict
from langgraph.graph import StateGraph, END

class ReportState(TypedDict):
    topic: str
    research_data: list
    report: str

def research_node(state: ReportState):
    # ... logic to research the topic ...
    print("---RESEARCHING---")
    data = ["AI helps code faster.", "Many new tools exist."] # Dummy data
    return {"research_data": data}

def write_node(state: ReportState):
    # ... logic to write the report ...
    print("---WRITING---")
    research = state["research_data"]
    return {"report": f"Report based on: {research}"}

workflow = StateGraph(ReportState)
workflow.add_node("researcher", research_node)
workflow.add_node("writer", write_node)
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", END)
workflow.set_entry_point("researcher")

app = workflow.compile()
final_state = app.invoke({"topic": "AI code generation tools"})
print(final_state['report'])
```

The contrast is stark. The CrewAI code reads like a project plan, focusing on *what* needs to be done. The Autogen snippet resembles setting up a chat room, focusing on *who* needs to talk. The LangGraph code looks like an engineering diagram, meticulously defining the *how*—the state and the exact sequence of operations. This difference in style is not just cosmetic; it directly reflects the trade-offs between simplicity, conversational flexibility, and procedural control that we discussed earlier.

---

### The Verdict: Which Framework Should You Choose?

The choice ultimately depends on your project's specific needs and your development philosophy. To make that decision easier, here is a clear guide based on common project goals and team priorities.

**Choose CrewAI if:**
*   You are new to multi-agent systems or want to prototype an idea **fast**.
*   Your problem can be naturally modeled as a team of specialists with clear roles and a defined process (e.g., Researcher -> Writer -> Editor).
*   You prioritize developer experience, readability, and speed over fine-grained, low-level control.

**Choose Autogen if:**
*   Your primary focus is on the complex, dynamic, and unpredictable nature of agent conversations.
*   You are conducting academic research, building social simulations, or exploring emergent agent behaviors.
*   You require sophisticated and flexible **human-in-the-loop** capabilities for moderation, teaching, or steering the agent collaboration.

**Choose LangGraph if:**
*   You need **absolute control** over the workflow, agent state, and transition logic between steps.
*   Your process is cyclical, involves many steps, and requires agents to reflect, self-correct, or retry tasks based on explicit conditions.
*   You are already heavily invested in the LangChain ecosystem and aim to build production-grade, observable, and debuggable **agentic workflows**.

---

### Conclusion

We've explored three powerful frameworks, each with a distinct identity for building the next generation of LLM applications. LangGraph offers ultimate **control** through its state machine paradigm. Autogen excels at orchestrating complex **conversations** and human-AI collaboration. CrewAI champions **simplicity** and rapid development with its intuitive role-playing metaphor.

There is no single "best" framework for every problem. The right choice is a function of your project's complexity, your need for control versus speed, and your team's existing expertise. The field of **multi-agent system frameworks** is evolving at an incredible pace, and the lines between these tools may blur over time as they borrow ideas from one another. The best way to make a decision is to get your hands dirty, experiment with a small project, and see which paradigm clicks for you.

---

**What are your experiences with these frameworks? Did I miss anything? Share your thoughts and projects in the comments below!**

*For more deep dives into AI engineering and agentic systems, follow me on Medium and connect on [Your LinkedIn Link].*

*You can find the illustrative code snippets for this article on this [Link to GitHub Gist or Repo].*

## Exercise

- Now change the LLM used to GPT3.5, generate again the blog and compare the results in terms of readibility.
  